In [1]:
# Imports

import os
import re
import pandas as pd
from tqdm.auto import tqdm
from rdkit import Chem
from multiprocessing import Pool, cpu_count
tqdm.pandas()

/data/ryanschen/safe-retro/saferetrouv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#Paths
DATA_DIR   = '/data/ryanschen/safe-retro/Data'  
TRAIN_FILE = 'train_safe.csv'
VAL_FILE   = 'val_safe.csv'
TEST_FILE  = 'test_safe.csv'

SRC_COL = 'products'    # plain SMILES product  (encoder input)
TGT_COL = 'precursors'  # plain SMILES precursors (decoder target)


def load_file(path):
    return pd.read_csv(path)

train_df = load_file(os.path.join(DATA_DIR, TRAIN_FILE))
val_df   = load_file(os.path.join(DATA_DIR, VAL_FILE))
test_df  = load_file(os.path.join(DATA_DIR, TEST_FILE))

print(f'Train : {len(train_df):,} reactions')
print(f'Val   : {len(val_df):,} reactions')
print(f'Test  : {len(test_df):,} reactions')
train_df[[SRC_COL, TGT_COL]].head(3)

Train : 409,006 reactions
Val   : 29,998 reactions
Test  : 39,994 reactions


,products,precursors
0,CC(C)CC(=O)c1ccc(O)nc1,C1CCOC1.CC(C)C[Mg+].CON(C)C(=O)c1ccc(O)nc1.[Cl-]
1,CNc1ccc(C(=O)O)cc1[N+](=O)[O-],CN.O.O=C(O)c1ccc(Cl)c([N+](=O)[O-])c1
2,CCn1cc(C(=O)O)c(=O)c2cc(F)c(-c3ccc(NC=O)cc3)cc21,CCn1cc(C(=O)O)c(=O)c2cc(F)c(-c3ccc(N)cc3)cc21....


In [ ]:
# Tokenization

SMI_REGEX_PATTERN = (
    r'(\%\([0-9]{3}\)|\[[^\]]+]|Br?|Cl?|N|O|S|P|F|I|b|c|n|o|s|p'
    r'|\||\(|\)|\.|=|#|-|\+|\\|\/|:|~|@|\?|>>?|\*|\$|\%[0-9]{2}|[0-9])'
)
_regex = re.compile(SMI_REGEX_PATTERN)

def tokenize(smiles: str) -> str:
    return ' '.join(_regex.findall(smiles))

# Quick demo
example = train_df[SRC_COL].iloc[0]
print('Original :', example)
print('Tokenized:', tokenize(example))


train_df['tok_src'] = train_df[SRC_COL].progress_apply(tokenize)
train_df['tok_tgt'] = train_df[TGT_COL].progress_apply(tokenize)


val_df['tok_src'] = val_df[SRC_COL].progress_apply(tokenize)
val_df['tok_tgt'] = val_df[TGT_COL].progress_apply(tokenize)

test_df['tok_src'] = test_df[SRC_COL].progress_apply(tokenize)
test_df['tok_tgt'] = test_df[TGT_COL].progress_apply(tokenize)

print('All good!')

Original : CC(C)CC(=O)c1ccc(O)nc1
Tokenized: C C ( C ) C C ( = O ) c 1 c c c ( O ) n c 1


100%|██████████| 39994/39994 [00:00<00:00, 198052.37it/s]

All good!


In [6]:
# Saving tokenized files

DATA_OUT = 'USPTO_SMILES_preprocessed'
os.makedirs(DATA_OUT, exist_ok=True)

train_shuffled = train_df.sample(frac=1.0, random_state=42)

splits = {
    'train': train_shuffled,
    'val'  : val_df,
    'test' : test_df,
}

for split_name, df in splits.items():
    for col, fname in [('tok_src', f'src-{split_name}.txt'),
                       ('tok_tgt', f'tgt-{split_name}.txt')]:
        path = os.path.join(DATA_OUT, fname)
        with open(path, 'w') as f:
            f.write('\n'.join(df[col].values))
        print(f'Saved {path}  ({len(df):,} lines)')


Saved USPTO_SMILES_preprocessed/src-train.txt  (409,006 lines)
Saved USPTO_SMILES_preprocessed/tgt-train.txt  (409,006 lines)
Saved USPTO_SMILES_preprocessed/src-val.txt  (29,998 lines)
Saved USPTO_SMILES_preprocessed/tgt-val.txt  (29,998 lines)
Saved USPTO_SMILES_preprocessed/src-test.txt  (39,994 lines)
Saved USPTO_SMILES_preprocessed/tgt-test.txt  (39,994 lines)


In [8]:
# Vocab of OpenNMT

CONFIG_DIR = 'smiles_run'
os.makedirs(CONFIG_DIR, exist_ok=True)

config_yaml = f"""
## OpenNMT-py config for SMILES Molecular Transformer baseline

save_data: {CONFIG_DIR}/data
src_vocab: {CONFIG_DIR}/smiles.vocab.src
tgt_vocab: {CONFIG_DIR}/smiles.vocab.src
overwrite: true
share_vocab: true

data:
    corpus-1:
        path_src: {DATA_OUT}/src-train.txt
        path_tgt: {DATA_OUT}/tgt-train.txt
    valid:
        path_src: {DATA_OUT}/src-val.txt
        path_tgt: {DATA_OUT}/tgt-val.txt

world_size: 1
gpu_ranks: [0]
save_model: {CONFIG_DIR}/model
save_checkpoint_steps: 5000
keep_checkpoint: 5
train_steps: 400000
valid_steps: 10000
report_every: 100
tensorboard: true
tensorboard_log_dir: smiles_log_dir
"""

config_path = os.path.join(CONFIG_DIR, 'run_config_smiles.yaml')
with open(config_path, 'w') as f:
    f.write(config_yaml)
print(f'Config written to {config_path}')

!onmt_build_vocab -config {config_path} \
    -src_seq_length 1000 -tgt_seq_length 1000 \
    -src_vocab_size 1000 -tgt_vocab_size 1000 \
    -n_sample -1

# Inspect vocab
vocab_path = os.path.join(CONFIG_DIR, 'smiles.vocab.src')
with open(vocab_path) as f:
    vocab = [line.split()[0] for line in f]
print(f'Vocabulary size: {len(vocab)} tokens')
print('First 30 tokens:', vocab[:30])

Config written to smiles_run/run_config_smiles.yaml
/data/ryanschen/safe-retro/saferetrouv/lib/python3.11/site-packages/onmt/modules/sparse_activations.py:47: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd
/data/ryanschen/safe-retro/saferetrouv/lib/python3.11/site-packages/onmt/modules/sparse_activations.py:67: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @custom_bwd
/data/ryanschen/safe-retro/saferetrouv/lib/python3.11/site-packages/onmt/modules/sparse_losses.py:12: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd
/data/ryanschen/safe-retro/saferetrouv/lib/python3.11/site-packages/onmt/modules/sparse_losses.py:36: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Plea